In [1]:
import joblib
import pandas as pd


#Load data
df_train = pd.read_csv('data/german_test.csv')
df_test = pd.read_csv('data/german_test.csv')
X_train, y_train = df_train.drop(columns=['target']), df_train['target']
X_test, y_test = df_test.drop(columns=['target']), df_test['target']

#Load model and encoders
model = joblib.load('models/xgboost_model_credit.pkl')
encoder = joblib.load('encoders/catboost_encoder_credit.pkl')
label = joblib.load('encoders/label_encoder_credit.pkl')

In [2]:
from celia import CELIA
from celia.explainers.dice import Dice, Data, Model
from test_constants import *

In [3]:
#Metadata
categorical_cols = GERMAN_CREDIT_CATEGORIC_COLUMNS
immutable_features = GERMAN_CREDIT_IMMUTABLE_FEATURES
mutable_features = X_train.columns.difference(immutable_features).tolist()
continuous_features = X_train.columns.difference(categorical_cols).tolist()
feasible_ranges = GERMAN_CREDIT_FEASIBLE_VALUES

In [4]:
d = Data(dataframe=df_train, continuous_features=continuous_features, outcome_name="target", permitted_range=feasible_ranges)
m = Model(model=model, backend="sklearn", model_type="classifier")
dice_random = Dice(d, m, method="random")
dice_genetic = Dice(d, m, method="genetic")

In [7]:
itbe = X_test.iloc[0].to_frame().T

In [8]:
results = dice_random.generate_counterfactuals(
            itbe,
            total_CFs=3,
            features_to_vary=mutable_features
        )

In [10]:
counterfactual_list = results.cf_examples_list[0].final_cfs_df

In [11]:
counterfactual_list

,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,...,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,target
0,0.602475,18.0,0.667188,0.641772,15515.8,0.818,0.740191,4.0,0.729291,0.699862,...,0.693158,42.0,0.726677,0.733966,2.0,0.708071,1.0,0.682708,0.693014,0
1,0.602475,18.0,0.667188,0.641772,13335.1,0.818,0.740191,4.0,0.729291,0.699862,...,0.693158,42.0,0.726677,0.733966,2.0,0.708071,1.0,0.682708,0.693014,0
2,0.602475,18.0,0.667188,0.641772,14479.4,0.818,0.740191,4.0,0.729291,0.699862,...,0.693158,42.0,0.726677,0.733966,2.0,0.708071,1.0,0.725776,0.693014,0


In [14]:
def check_feasibility(counterfactuals_df, feasible_values):
    violations = {}

    for feature, constraint in feasible_values.items():
        if feature not in counterfactuals_df.columns:
            continue  # or raise an error if strict checking is preferred

        values = counterfactuals_df[feature]

        if isinstance(constraint, tuple) and len(constraint) == 2:
            # Continuous/range constraint
            lower, upper = constraint
            invalid_mask = (values < lower) | (values > upper)

        elif isinstance(constraint, list):
            # Categorical constraint
            invalid_mask = ~values.isin(constraint)

        else:
            raise ValueError(f"Unsupported constraint type for feature '{feature}': {constraint}")

        if invalid_mask.any():
            violations[feature] = values[invalid_mask]

    return violations


In [15]:
violations = check_feasibility(counterfactual_list, GERMAN_CREDIT_FEASIBLE_VALUES)

if violations:
    print("Some counterfactuals have infeasible values:")
    for feature, invalid_values in violations.items():
        print(f"- {feature}: {invalid_values.tolist()}")
else:
    print("All counterfactuals are feasible.")


All counterfactuals are feasible.
